# NLU Gabungan HOSTAGE: Intent dan Target dalam Satu IndoBERT

Notebook ini melatih **satu model IndoBERT** yang menebak intent pesan (offend/defend/neutral) **dan** relasi tiap pemain (target), lalu membandingkannya dengan **pipeline dua model** yang dipakai bot sekarang:

| Sistem | Model | Cara kerja |
| --- | --- | --- |
| Pipeline dua model | Model 1 (`nlu_baru`) + Model 2 (`nlu_target`) | Model 1 menebak intent dari teks. Model 2 menerima intent itu di awal teks pasangan (`intent offend || ...`) lalu menebak relasi tiap kandidat. Kesalahan Model 1 terbawa ke Model 2. |
| **Model gabungan** (notebook ini) | Satu encoder IndoBERT dengan dua kepala | Kepala intent membaca teks pesan. Kepala relasi membaca teks pasangan **tanpa** awalan intent, jadi relasi ditebak sendiri. Kedua kepala dilatih bersamaan dan berbagi encoder. |

Data, pembagian train/validation/test, teks pasangan, ruang learning rate, dan aturan seleksi **sama persis** dengan `nlu_baru` dan `nlu_target`. Perbedaan hasil berasal dari arsitekturnya, bukan dari data atau parameter. Label memakai aturan `data/ATURAN_LABEL.md` (relabel 26 Oktober 2026).

Perbandingan dibuat **ujung ke ujung**: intent pada pipeline berasal dari tebakan Model 1, bukan dari kunci jawaban, karena di game intent juga hasil tebakan.

## Library yang digunakan

In [1]:
# Library standar Python: file, waktu, pengacakan, dan memori.
import copy
import gc
import json
import math
import random
import re
from collections import Counter
from functools import lru_cache
from pathlib import Path
from time import perf_counter

# Pengolahan data dan tampilan tabel.
import numpy as np
import pandas as pd
from IPython.display import display

# Pembagian data dan metrik evaluasi (sama dengan nlu_baru dan nlu_target).
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

# IndoBERT: encoder bersama, tokenizer, penjadwal learning rate, dan penyimpanan bobot.
import torch
import transformers
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import (
    AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
    get_linear_schedule_with_warmup, set_seed,
)

C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pengaturan eksperimen

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `DATASET_INTENT / DATASET_TARGET` | Dataset Model 1 dan Model 2. | Sama dengan yang dipakai `nlu_baru` dan `nlu_target`. |
| `MODEL_1_DIR / MODEL_2_DIR` | Model pipeline yang sudah dilatih. | Dipakai hanya di bagian perbandingan; jalankan `nlu_baru` dan `nlu_target` lebih dulu. |
| `MODEL_GABUNGAN_DIR` | Tempat menyimpan model gabungan. | Berisi bobot (`model.safetensors`), konfigurasi encoder, tokenizer, label, dan metrik. |
| `RANDOM_STATE=42` | Seed pembagian data dan training. | Sama dengan kedua notebook lain sehingga test set-nya identik. |
| `JUMLAH_PEMAIN_MINIMUM=6` | Roster dilengkapi pemain semu. | Sama dengan `nlu_target`; pemain semu selalu `tidak_ada`. |
| `LATIH_GABUNGAN` | Saklar training. | False: training dilewati dan model tersimpan dimuat dari `MODEL_GABUNGAN_DIR`. |

In [2]:
# Dataset yang sama dengan Model 1 (nlu_baru) dan Model 2 (nlu_target).
DATASET_INTENT = Path("data/dataset_final3.csv")
DATASET_TARGET = Path("data/dataset_final3_context_target_pilot25_lengkap.csv")

# Model pipeline (pembanding) dan lokasi model gabungan.
MODEL_1_DIR = Path("models/notebook_standalone/intent_classifier_transformer")
MODEL_2_DIR = Path("models/target_classifier/target_classifier_transformer")
MODEL_GABUNGAN_DIR = Path("models/gabungan/nlu_gabungan_transformer")

# Seed dan roster minimum sama dengan nlu_baru dan nlu_target.
RANDOM_STATE = 42
JUMLAH_PEMAIN_MINIMUM = 6

# Saklar training: True = dilatih pada run ini, False = memuat model tersimpan.
LATIH_GABUNGAN = True

## Data

### Membaca Dataset

Kedua loader disalin dari notebook asalnya tanpa perubahan, supaya pembersihan dan validasi datanya identik.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `load_clean_nlu_dataset` | Dataset intent (Model 1). | Spasi dirapikan, duplikat persis dibuang, teks yang punya dua label berbeda dibuang. |
| `load_target_dataset` | Dataset target (Model 2). | Kolom JSON menjadi list; baris yang melanggar aturan target (mis. neutral berelasi offend) membuat notebook berhenti. |

In [3]:
# Membaca CSV intent, membersihkan chat dan label, lalu mengembalikan data bersih serta laporannya (sama dengan nlu_baru).
def load_clean_nlu_dataset(dataset_path):
    path = Path(dataset_path)
    if not path.is_file():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {path}")
    raw = pd.read_csv(path)
    report = {"dataset_path": str(path), "rows_raw": len(raw)}

    # Merapikan spasi dan label, lalu membuang baris kosong, duplikat persis, dan teks berlabel ganda.
    clean = raw[["teks_chat", "label_intent"]].dropna().copy()
    clean["teks_chat"] = clean["teks_chat"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    clean["label_intent"] = clean["label_intent"].astype(str).str.strip().str.lower()
    clean = clean.loc[(clean["teks_chat"] != "") & (clean["label_intent"] != "")]
    report["exact_duplicates_removed"] = int(clean.duplicated(["teks_chat", "label_intent"]).sum())
    clean = clean.drop_duplicates(["teks_chat", "label_intent"])
    label_count_per_text = clean.groupby("teks_chat")["label_intent"].nunique()
    conflicting_texts = label_count_per_text[label_count_per_text > 1].index
    report["conflicting_texts_removed"] = int(len(conflicting_texts))
    clean = clean.loc[~clean["teks_chat"].isin(conflicting_texts)]
    report.update({"rows_clean": len(clean), "class_distribution": clean["label_intent"].value_counts().sort_index().to_dict()})
    return clean.reset_index(drop=True), report


# Kolom target yang disimpan sebagai teks JSON di CSV.
JSON_COLUMNS = ["chat_sebelumnya", "daftar_pemain", "target"]


# Membaca CSV target, mengubah kolom JSON, lalu memeriksa konsistensi setiap baris (sama dengan nlu_target).
def load_target_dataset(dataset_path):
    path = Path(dataset_path)
    if not path.is_file():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {path}")
    data = pd.read_csv(path, keep_default_na=False)
    for column in JSON_COLUMNS:
        data[column] = data[column].apply(json.loads)
    data["teks_chat"] = data["teks_chat"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    data["label_intent"] = data["label_intent"].astype(str).str.strip().str.lower()

    # Pengirim dan target harus di roster; neutral hanya berelasi neutral; offend/defend wajib bertarget.
    problems = []
    for index, row in data.iterrows():
        players = set(row["daftar_pemain"])
        relasi = {t["relasi"] for t in row["target"]}
        if row["pengirim"] not in players:
            problems.append((index, "pengirim di luar daftar_pemain"))
        if any(t["pemain"] not in players and t["pemain"] != "tidak_diketahui" for t in row["target"]):
            problems.append((index, "target di luar daftar_pemain"))
        if row["label_intent"] == "neutral" and relasi - {"neutral"}:
            problems.append((index, "target pesan neutral harus berelasi neutral"))
        if row["label_intent"] != "neutral" and (not row["target"] or "neutral" in relasi):
            problems.append((index, "offend/defend wajib bertarget dan tidak berelasi neutral"))
    if problems:
        raise ValueError(f"{len(problems)} baris tidak konsisten, contoh: {problems[:3]}")
    report = {"dataset_path": str(path), "rows": len(data),
              "intent_distribution": data["label_intent"].value_counts().sort_index().to_dict()}
    return data.reset_index(drop=True), report


# Langsung baca kedua dataset.
data_intent, laporan_intent = load_clean_nlu_dataset(DATASET_INTENT)
data_target, laporan_target = load_target_dataset(DATASET_TARGET)
print("Intent:", laporan_intent["rows_clean"], "chat", laporan_intent["class_distribution"])
print("Target:", laporan_target["rows"], "pesan", laporan_target["intent_distribution"])

Intent: 5611 chat {'defend': 1048, 'neutral': 1934, 'offend': 2629}
Target: 775 pesan {'defend': 323, 'neutral': 130, 'offend': 322}


### Penyamaran Nama dan Teks Pasangan

Fungsi di bawah disalin dari `nlu_target` tanpa perubahan, kecuali satu hal: `bentuk_pasangan` sekarang bisa **tanpa** awalan intent.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `samarkan_nama` | Mengganti nama pemain dengan `KANDIDAT_X`, `PENGIRIM_X`, `PEMAIN_X`. | Model belajar pola kalimat, bukan menghafal nama. |
| `buat_teks_pasangan` | Teks satu pasangan (pesan, kandidat) beserta 12 konteks terbaru dan anotasinya. | `label_intent=None` berarti tanpa awalan `intent ... ||` (dipakai model gabungan). |
| `bentuk_pasangan(intent_awalan=...)` | Satu baris per (pesan, kandidat) berlabel relasi. | `None`: tanpa awalan (gabungan). Dict `row_id → intent`: awalan intent tertentu (pipeline: dari tebakan Model 1). |
| `RELASI_LABELS` | Kelas kepala relasi. | `defend`, `neutral`, `offend`, `tidak_ada`; urutan sama dengan Model 2. |

In [4]:
# Token pengganti nama pemain dan label relasi (sama dengan nlu_target).
TOKEN_KANDIDAT = "KANDIDAT_X"
TOKEN_PENGIRIM = "PENGIRIM_X"
TOKEN_PEMAIN = "PEMAIN_X"
TOKEN_TIDAK_DIKETAHUI = "TIDAK_DIKETAHUI"
RELASI_LABELS = ["defend", "neutral", "offend", "tidak_ada"]
INTENT_LABELS = ["defend", "neutral", "offend"]

# Kata ganti dan istilah game tidak pernah dianggap nama pemain.
KATA_GANTI = {
    "aku", "saya", "gw", "gue", "gua", "dia", "kamu", "kau", "lu", "lo", "elu",
    "anda", "mereka", "kalian", "kita", "kami", "semua", "si",
}
ISTILAH_GAME = {"hitman", "spy", "stalker", "civilian", "hostage", "gag", "order", "guard", "peek", "vote"}
AKHIRAN_NAMA = ("nya", "lah", "kah", "pun", "mu", "ku")
NAMA_SEMU = ["Agus", "Bella", "Candra", "Desi", "Erik", "Fitri", "Galang", "Hesti", "Irwan", "Jessi"]


# Kata yang sering muncul di chat asli (huruf kecil di tengah kalimat) tidak dikira nama panggilan.
def build_common_words(source_path, min_count=3):
    counts, total, kapital = Counter(), Counter(), Counter()
    for text in pd.read_csv(source_path)["teks_chat"].dropna().astype(str):
        counts.update(set(re.findall(r"\w+", text.casefold())))
        for index, match in enumerate(re.finditer(r"\w+", text)):
            if index == 0 or text[:match.start()].rstrip().endswith((".", "!", "?")):
                continue
            key = match.group(0).casefold()
            total[key] += 1
            kapital[key] += match.group(0)[:1].isupper()
    return frozenset(
        word for word, count in counts.items()
        if count >= min_count and len(word) >= 2 and kapital[word] <= total[word] * 0.5
    )


# Potongan nama (Dika untuk Andika) atau nama berakhiran (Budinya, Andyy).
@lru_cache(maxsize=None)
def _nama_cocok(word, name):
    w, n = word.casefold(), name.casefold()
    if len(w) < 3:
        return False
    if w in n:
        return True
    sisa = w[len(n):] if w.startswith(n) else None
    return sisa is not None and (sisa in AKHIRAN_NAMA or set(sisa) == {n[-1]})


# Mengganti nama pemain dan panggilannya dengan token peran.
def samarkan_nama(text, daftar_pemain, kandidat, pengirim, common_words):
    lowered = {name.casefold(): name for name in daftar_pemain}

    def token_for(name):
        if name == kandidat:
            return TOKEN_KANDIDAT
        return TOKEN_PENGIRIM if name == pengirim else TOKEN_PEMAIN

    def replace(match):
        word = match.group(0)
        key = word.casefold()
        if key in lowered:
            return token_for(lowered[key])
        if key in KATA_GANTI or key in ISTILAH_GAME or key in common_words:
            return word
        matches = [name for name in daftar_pemain if _nama_cocok(word, name)]
        return token_for(matches[0]) if len(matches) == 1 else word

    return re.sub(r"\w+", replace, text)


# Satu teks pasangan (pesan, kandidat): pesan sekarang lalu konteks terbaru lebih dulu beserta anotasinya.
def buat_teks_pasangan(pengirim, teks_chat, chat_sebelumnya, daftar_pemain, kandidat, common_words, label_intent=None):
    def peran(name):
        if name == "tidak_diketahui":
            return TOKEN_TIDAK_DIKETAHUI
        if name == kandidat:
            return TOKEN_KANDIDAT
        return TOKEN_PENGIRIM if name == pengirim else TOKEN_PEMAIN

    bagian = []
    for pesan in reversed(chat_sebelumnya):
        relasi = ", ".join(f"{t['relasi']} {peran(t['pemain'])}" for t in pesan["target"])
        anotasi = f"{pesan['label_intent']}: {relasi}" if relasi else pesan["label_intent"]
        teks = samarkan_nama(pesan["teks_chat"], daftar_pemain, kandidat, pengirim, common_words)
        bagian.append(f"{peran(pesan['pengirim'])}: {teks} [{anotasi}]")
    sekarang = samarkan_nama(teks_chat, daftar_pemain, kandidat, pengirim, common_words)
    konteks = " | ".join(bagian) if bagian else "tidak ada"
    awal = f"intent {label_intent} || " if label_intent else ""
    return f"{awal}pesan sekarang {peran(pengirim)}: {sekarang} || konteks: {konteks}"


# Satu baris per (pesan, kandidat). intent_awalan: None = tanpa awalan, dict row_id → intent = awalan itu.
def bentuk_pasangan(data, common_words, jumlah_pemain_minimum=0, intent_awalan=None):
    rows = []
    for row_id, row in data.iterrows():
        gold = {t["pemain"]: t["relasi"] for t in row["target"] if t["pemain"] != "tidak_diketahui"}
        kandidat_list = list(row["daftar_pemain"])
        semu = [name for name in NAMA_SEMU if name.casefold() not in {p.casefold() for p in kandidat_list}]
        kandidat_list += semu[:max(0, jumlah_pemain_minimum - len(kandidat_list))]
        awalan = None if intent_awalan is None else intent_awalan[row_id]
        for kandidat in kandidat_list:
            rows.append({
                "row_id": row_id, "case_name": row["case_name"], "label_intent": row["label_intent"],
                "kandidat": kandidat,
                "teks": buat_teks_pasangan(row["pengirim"], row["teks_chat"], row["chat_sebelumnya"],
                                           row["daftar_pemain"], kandidat, common_words, awalan),
                "relasi": gold.get(kandidat, "tidak_ada"),
            })
    return pd.DataFrame(rows)


# Langsung siapkan kata umum dan tampilkan satu contoh teks pasangan tanpa awalan intent.
COMMON_WORDS = build_common_words(DATASET_INTENT)
print(f"Kata umum dari dataset sumber: {len(COMMON_WORDS)}")
print(buat_teks_pasangan("Andi", "Aku curiga Budi.", [], ["Andi", "Budi", "Citra"], "Budi", COMMON_WORDS))

Kata umum dari dataset sumber: 1185
pesan sekarang PENGIRIM_X: Aku curiga KANDIDAT_X. || konteks: tidak ada


### Pembagian Train/Validation/Test

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `bagi_intent` | Split Model 1. | Sama dengan `nlu_baru`: test 20% stratifikasi intent, lalu validation 15% dari seluruh data. Test intent model gabungan = test Model 1. |
| `bagi_target` | Split Model 2. | Sama dengan `nlu_target`: `StratifiedGroupKFold` per grup kalimat sumber, strata kasus; validation diambil dengan cara yang sama dari pool train. |
| `pasangan[...]` | Pasangan kandidat tanpa awalan intent. | Kepala relasi model gabungan dilatih dan diuji dengan teks ini. |

In [5]:
# Model 1: test 20% dan validation 15% dengan stratifikasi intent (sama dengan nlu_baru).
def bagi_intent(data, validation_size=0.15):
    x_pool, x_test, y_pool, y_test = train_test_split(
        data["teks_chat"], data["label_intent"], test_size=0.2, random_state=RANDOM_STATE, stratify=data["label_intent"])
    x_train, x_val, y_train, y_val = train_test_split(
        x_pool, y_pool, test_size=validation_size / 0.8, random_state=RANDOM_STATE, stratify=y_pool)
    return {"train": (x_train, y_train), "validation": (x_val, y_val), "test": (x_test, y_test)}


# Model 2: pesan dibagi per grup kalimat sumber dengan kasus terwakili (sama dengan nlu_target).
def _split_rows(data, test_size=0.2):
    splitter = StratifiedGroupKFold(n_splits=round(1 / test_size), shuffle=True, random_state=RANDOM_STATE)
    train_index, test_index = next(splitter.split(data, data["case_name"], groups=data["source_group"]))
    return data.iloc[train_index], data.iloc[test_index]


def bagi_target(data, validation_size=0.15):
    pool_rows, test_rows = _split_rows(data, 0.2)
    splitter = StratifiedGroupKFold(n_splits=max(2, round(0.8 / validation_size)), shuffle=True, random_state=RANDOM_STATE)
    train_index, val_index = next(splitter.split(pool_rows, pool_rows["case_name"], groups=pool_rows["source_group"]))
    return {"train": pool_rows.iloc[train_index], "validation": pool_rows.iloc[val_index], "test": test_rows}


# Langsung bagi kedua dataset dan bentuk pasangan tanpa awalan intent.
split_intent = bagi_intent(data_intent)
split_target = bagi_target(data_target)
pasangan = {nama: bentuk_pasangan(rows, COMMON_WORDS, JUMLAH_PEMAIN_MINIMUM) for nama, rows in split_target.items()}
display(pd.DataFrame([
    {"bagian": nama, "chat intent": len(split_intent[nama][0]), "pesan target": len(split_target[nama]),
     "pasangan relasi": len(pasangan[nama])}
    for nama in ("train", "validation", "test")
]).style.hide(axis="index"))

bagian,chat intent,pesan target,pasangan relasi
train,3646,496,2977
validation,842,124,745
test,1123,155,931


## Arsitektur Model Gabungan

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `encoder` | IndoBERT (`indobenchmark/indobert-base-p1`) yang dipakai kedua tugas. | Satu set bobot ±124 juta parameter, bukan dua. |
| `pooler_output` + `dropout` | Ringkasan kalimat dari token `[CLS]`. | Sama dengan cara `AutoModelForSequenceClassification` (dipakai Model 1 dan 2). |
| `kepala_intent` | Lapisan linear 768 → 3. | Membaca teks pesan saja (seperti Model 1). |
| `kepala_relasi` | Lapisan linear 768 → 4. | Membaca teks pasangan tanpa awalan intent. |
| `loss` | Cross-entropy kepala yang sesuai tugas batch. | Setiap batch berisi satu tugas; batch intent dan relasi diselang-seling sepanjang epoch sehingga encoder belajar keduanya. |

In [6]:
# Satu encoder IndoBERT dengan dua kepala klasifikasi.
class ModelGabungan(nn.Module):
    def __init__(self, encoder, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.kepala_intent = nn.Linear(hidden, len(INTENT_LABELS))
        self.kepala_relasi = nn.Linear(hidden, len(RELASI_LABELS))

    # tugas = "intent" atau "relasi"; keluarannya logit kelas untuk tugas itu.
    def forward(self, input_ids, attention_mask, token_type_ids=None, tugas="intent"):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                              token_type_ids=token_type_ids).pooler_output
        kepala = self.kepala_intent if tugas == "intent" else self.kepala_relasi
        return kepala(self.dropout(pooled))

    # Membuat model baru dari IndoBERT pralatih.
    @classmethod
    def dari_pralatih(cls, model_name, dropout):
        encoder = AutoModel.from_pretrained(model_name, hidden_dropout_prob=dropout,
                                            attention_probs_dropout_prob=dropout, local_files_only=True)
        return cls(encoder, dropout)

    # Menyimpan bobot (safetensors), konfigurasi encoder, dan info label.
    def simpan(self, folder, info):
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        self.encoder.config.save_pretrained(folder)
        save_file({k: v.contiguous().cpu() for k, v in self.state_dict().items()}, str(folder / "model.safetensors"))
        (folder / "gabungan_config.json").write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")

    # Memuat model tersimpan tanpa mengunduh apa pun.
    @classmethod
    def muat(cls, folder):
        folder = Path(folder)
        model = cls(AutoModel.from_config(AutoConfig.from_pretrained(folder, local_files_only=True)))
        model.load_state_dict(load_file(str(folder / "model.safetensors")))
        return model.eval()

## Training

### Parameter Training

Nilainya mengikuti Model 1 dan Model 2 supaya perbandingannya adil.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `learning_rates` | Pilihan langkah pembaruan bobot. | 1e-5, 2e-5, 3e-5 (sama dengan kedua model); pemenang dipilih dari validation. |
| `epochs=12`, `early_stopping_patience=3` | Batas putaran dan toleransi tanpa perbaikan. | Berhenti setelah 3 epoch tanpa kenaikan skor validation > 0.001. |
| `batch_efektif=16` | Contoh per pembaruan bobot. | Intent: 2 × 8 chat (seperti Model 1). Relasi: 4 × 4 pasangan (seperti Model 2, agar 512 token muat di GPU 6 GB). |
| `max_length_intent=128 / max_length_relasi=512` | Batas token per tugas. | Sama dengan Model 1 dan Model 2. |
| `weight_decay=0.01`, `warmup_ratio=0.1`, `dropout=0.1`, `max_grad_norm=1.0` | Regularisasi dan jadwal learning rate. | Sama dengan kedua model. |
| `skor validation` | Dasar memilih epoch dan learning rate. | Rata-rata macro-F1 intent dan macro-F1 relasi pada validation. Test tidak pernah dipakai untuk memilih. |

In [7]:
# Pengaturan training model gabungan.
PENGATURAN = {
    "model_name": "indobenchmark/indobert-base-p1",
    "device": "cuda",  # Ganti cpu jika GPU CUDA tidak tersedia.
    "learning_rates": (1e-5, 2e-5, 3e-5),
    "epochs": 12,
    "batch_efektif": 16,
    "micro_batch": {"intent": 8, "relasi": 4},
    "max_length": {"intent": 128, "relasi": 512},
    "eval_batch_size": 16,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout": 0.1,
    "max_grad_norm": 1.0,
    "early_stopping_patience": 3,
    "early_stopping_threshold": 0.001,
}

# Presisi otomatis seperti kedua notebook: BF16 bila didukung, selain itu FP16 di CUDA, FP32 di CPU.
DEVICE = torch.device(PENGATURAN["device"] if torch.cuda.is_available() else "cpu")
PRESISI = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
           else torch.float16 if DEVICE.type == "cuda" else None)
tokenizer = AutoTokenizer.from_pretrained(PENGATURAN["model_name"], local_files_only=True)
print("Perangkat:", DEVICE, "| presisi:", PRESISI or "fp32")

Perangkat: cuda | presisi: torch.bfloat16


### Data Training per Tugas

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `DATA_TUGAS[tugas][bagian]` | Teks dan label angka untuk setiap tugas. | Intent: chat dari `dataset_final3.csv`. Relasi: teks pasangan dari dataset target. |
| `jadwal_batch` | Urutan batch dalam satu epoch. | Semua batch intent dan relasi dikocok bersama, sehingga setiap epoch melewati seluruh data kedua tugas. |
| `peluang` | Probabilitas kelas untuk sekumpulan teks. | Dipakai di validation, test, dan perbandingan. |

In [8]:
# Teks dan label angka untuk kedua tugas.
DATA_TUGAS = {
    "intent": {nama: (list(x), [INTENT_LABELS.index(y) for y in ys]) for nama, (x, ys) in split_intent.items()},
    "relasi": {nama: (frame["teks"].tolist(), [RELASI_LABELS.index(r) for r in frame["relasi"]])
               for nama, frame in pasangan.items()},
}


# Mengubah sekumpulan teks menjadi tensor token di perangkat training.
def token(teks, tugas):
    encoded = tokenizer(list(teks), truncation=True, max_length=PENGATURAN["max_length"][tugas],
                        padding=True, return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in encoded.items()}


# Urutan batch satu epoch: batch kedua tugas dikocok bersama dengan seed per epoch.
def jadwal_batch(epoch):
    rng = random.Random(RANDOM_STATE + epoch)
    jadwal = []
    for tugas in ("intent", "relasi"):
        urutan = list(range(len(DATA_TUGAS[tugas]["train"][0])))
        rng.shuffle(urutan)
        jadwal += [(tugas, urutan[k:k + PENGATURAN["batch_efektif"]])
                   for k in range(0, len(urutan), PENGATURAN["batch_efektif"])]
    rng.shuffle(jadwal)
    return jadwal


# Probabilitas kelas untuk sekumpulan teks (tanpa gradien).
@torch.inference_mode()
def peluang(model, teks, tugas):
    model.eval()
    hasil = []
    for k in range(0, len(teks), PENGATURAN["eval_batch_size"]):
        with torch.autocast(DEVICE.type, dtype=PRESISI, enabled=PRESISI is not None):
            logits = model(**token(teks[k:k + PENGATURAN["eval_batch_size"]], tugas), tugas=tugas)
        hasil.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(hasil) if hasil else np.zeros((0, len(INTENT_LABELS if tugas == "intent" else RELASI_LABELS)))


# Skor validation: rata-rata macro-F1 intent dan macro-F1 relasi.
def skor_validation(model):
    hasil = {}
    for tugas in ("intent", "relasi"):
        teks, y = DATA_TUGAS[tugas]["validation"]
        prediksi = peluang(model, teks, tugas).argmax(axis=1)
        hasil[f"val_f1_{tugas}"] = float(f1_score(y, prediksi, average="macro", zero_division=0))
    hasil["val_skor"] = (hasil["val_f1_intent"] + hasil["val_f1_relasi"]) / 2
    return hasil


print({tugas: {nama: len(v[0]) for nama, v in bagian.items()} for tugas, bagian in DATA_TUGAS.items()})

{'intent': {'train': 3646, 'validation': 842, 'test': 1123}, 'relasi': {'train': 2977, 'validation': 745, 'test': 931}}


### Melatih Kandidat Learning Rate

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `latih_satu_lr` | Melatih model dari IndoBERT pralatih dengan satu learning rate. | Bobot terbaik (skor validation tertinggi) disimpan di memori CPU; epoch tanpa perbaikan > 0.001 dihitung untuk early stopping. |
| `riwayat` | Loss train dan skor validation per epoch. | Ditampilkan di tabel epoch di bawah. |
| `hasil_trial` | Ringkasan tiap learning rate. | Pemenang = skor validation tertinggi. |

In [9]:
# Melatih satu learning rate dan mengembalikan bobot terbaik beserta riwayat epoch.
def latih_satu_lr(rate):
    set_seed(RANDOM_STATE)
    model = ModelGabungan.dari_pralatih(PENGATURAN["model_name"], PENGATURAN["dropout"]).to(DEVICE)
    tanpa_decay = ("bias", "LayerNorm.weight")
    kelompok = [
        {"params": [p for n, p in model.named_parameters() if not n.endswith(tanpa_decay)],
         "weight_decay": PENGATURAN["weight_decay"]},
        {"params": [p for n, p in model.named_parameters() if n.endswith(tanpa_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(kelompok, lr=rate)
    langkah_total = PENGATURAN["epochs"] * len(jadwal_batch(0))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(PENGATURAN["warmup_ratio"] * langkah_total), langkah_total)
    scaler = torch.amp.GradScaler(enabled=PRESISI == torch.float16)

    terbaik, riwayat, tanpa_perbaikan = {"val_skor": -1.0}, [], 0
    for epoch in range(1, PENGATURAN["epochs"] + 1):
        model.train()
        mulai, total_loss = perf_counter(), {"intent": [], "relasi": []}
        for tugas, indeks in jadwal_batch(epoch):
            teks, label = DATA_TUGAS[tugas]["train"]
            optimizer.zero_grad()
            micro = PENGATURAN["micro_batch"][tugas]
            for k in range(0, len(indeks), micro):
                bagian = indeks[k:k + micro]
                with torch.autocast(DEVICE.type, dtype=PRESISI, enabled=PRESISI is not None):
                    logits = model(**token([teks[i] for i in bagian], tugas), tugas=tugas)
                y = torch.tensor([label[i] for i in bagian], device=DEVICE)
                # Loss dirata-rata per contoh di seluruh batch efektif (setara gradient accumulation).
                loss = nn.functional.cross_entropy(logits.float(), y) * len(bagian) / len(indeks)
                scaler.scale(loss).backward()
                total_loss[tugas].append(loss.item() * len(indeks) / len(bagian))
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), PENGATURAN["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        val = skor_validation(model)
        baris = {"epoch": epoch, "loss_intent": float(np.mean(total_loss["intent"])),
                 "loss_relasi": float(np.mean(total_loss["relasi"])), **val, "detik": round(perf_counter() - mulai, 1)}
        riwayat.append(baris)
        print(f"lr={rate:g} epoch {epoch}: loss intent {baris['loss_intent']:.4f} relasi {baris['loss_relasi']:.4f} | "
              f"val F1 intent {val['val_f1_intent']:.4f} relasi {val['val_f1_relasi']:.4f} skor {val['val_skor']:.4f} "
              f"| {baris['detik']} dtk", flush=True)

        # Bobot terbaik selalu disimpan; early stopping hanya menghitung perbaikan di atas ambang.
        perbaikan = val["val_skor"] - terbaik["val_skor"]
        if perbaikan > 0:
            terbaik = {**val, "epoch": epoch, "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
        tanpa_perbaikan = 0 if perbaikan > PENGATURAN["early_stopping_threshold"] else tanpa_perbaikan + 1
        if tanpa_perbaikan >= PENGATURAN["early_stopping_patience"]:
            break

    del model, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return terbaik, riwayat


# Langsung latih semua learning rate dan pilih pemenang dari validation.
if LATIH_GABUNGAN:
    hasil_trial, mulai_training = [], perf_counter()
    for rate in PENGATURAN["learning_rates"]:
        terbaik, riwayat = latih_satu_lr(rate)
        hasil_trial.append({"learning_rate": rate, "terbaik": terbaik, "riwayat": riwayat})
    durasi_training = perf_counter() - mulai_training
    pemenang = max(hasil_trial, key=lambda t: t["terbaik"]["val_skor"])
    model_gabungan = ModelGabungan.dari_pralatih(PENGATURAN["model_name"], PENGATURAN["dropout"])
    model_gabungan.load_state_dict(pemenang["terbaik"]["state"])
    model_gabungan.to(DEVICE).eval()
    display(pd.DataFrame([{"Learning rate": t["learning_rate"], "Epoch terbaik": t["terbaik"]["epoch"],
                           "Epoch dijalankan": len(t["riwayat"]), "Val F1 intent": t["terbaik"]["val_f1_intent"],
                           "Val F1 relasi": t["terbaik"]["val_f1_relasi"], "Val skor": t["terbaik"]["val_skor"]}
                          for t in hasil_trial]).style.format(precision=4).hide(axis="index"))
    print(f"Pemenang lr={pemenang['learning_rate']:g}, durasi training {durasi_training / 60:.1f} menit")
else:
    model_gabungan = ModelGabungan.muat(MODEL_GABUNGAN_DIR).to(DEVICE)
    print("Training dilewati: LATIH_GABUNGAN = False; model dimuat dari", MODEL_GABUNGAN_DIR)

lr=1e-05 epoch 1: loss intent 0.7198 relasi 0.6017 | val F1 intent 0.8375 relasi 0.3769 skor 0.6072 | 105.5 dtk


lr=1e-05 epoch 2: loss intent 0.3152 relasi 0.3011 | val F1 intent 0.8908 relasi 0.5779 skor 0.7344 | 108.8 dtk


lr=1e-05 epoch 3: loss intent 0.1826 relasi 0.2314 | val F1 intent 0.8999 relasi 0.7182 skor 0.8090 | 112.3 dtk


lr=1e-05 epoch 4: loss intent 0.1127 relasi 0.1542 | val F1 intent 0.8980 relasi 0.8289 skor 0.8634 | 102.0 dtk


lr=1e-05 epoch 5: loss intent 0.0540 relasi 0.0980 | val F1 intent 0.9046 relasi 0.8634 skor 0.8840 | 109.2 dtk


lr=1e-05 epoch 6: loss intent 0.0282 relasi 0.0587 | val F1 intent 0.9089 relasi 0.8682 skor 0.8885 | 112.2 dtk


... (19 outputs left out)

lr=3e-05 epoch 7: loss intent 0.0101 relasi 0.0351 | val F1 intent 0.9150 relasi 0.8620 skor 0.8885 | 109.8 dtk


lr=3e-05 epoch 8: loss intent 0.0021 relasi 0.0266 | val F1 intent 0.9094 relasi 0.8936 skor 0.9015 | 105.2 dtk


Learning rate,Epoch terbaik,Epoch dijalankan,Val F1 intent,Val F1 relasi,Val skor
0.0000,8,11,0.9071,0.9239,0.9155
0.0000,5,8,0.9168,0.8723,0.8946
0.0000,5,8,0.9080,0.8982,0.9031


Pemenang lr=1e-05, durasi training 51.1 menit


### Tabel Epoch

Tabel merangkum riwayat yang sudah tercatat; tidak melatih ulang.

In [10]:
# Riwayat per epoch untuk setiap learning rate.
if LATIH_GABUNGAN:
    for trial in hasil_trial:
        print(f"\nLearning rate {trial['learning_rate']:.0e}")
        display(pd.DataFrame(trial["riwayat"]).style.format(precision=4).hide(axis="index"))
else:
    print("Tabel epoch dilewati karena model tidak dilatih pada run ini.")


Learning rate 1e-05


epoch,loss_intent,loss_relasi,val_f1_intent,val_f1_relasi,val_skor,detik
1,0.7198,0.6017,0.8375,0.3769,0.6072,105.5000
2,0.3152,0.3011,0.8908,0.5779,0.7344,108.8000
3,0.1826,0.2314,0.8999,0.7182,0.8090,112.3000
4,0.1127,0.1542,0.8980,0.8289,0.8634,102.0000
5,0.0540,0.0980,0.9046,0.8634,0.8840,109.2000
6,0.0282,0.0587,0.9089,0.8682,0.8885,112.2000
7,0.0106,0.0383,0.9145,0.8772,0.8959,123.0000
8,0.0040,0.0273,0.9071,0.9239,0.9155,129.3000
9,0.0051,0.0213,0.9054,0.9021,0.9038,136.7000
10,0.0006,0.0132,0.9076,0.9030,0.9053,121.1000



Learning rate 2e-05


epoch,loss_intent,loss_relasi,val_f1_intent,val_f1_relasi,val_skor,detik
1,0.6303,0.5487,0.8458,0.3852,0.6155,110.9000
2,0.2972,0.3146,0.8667,0.5309,0.6988,106.3000
3,0.1651,0.2043,0.8971,0.8348,0.8659,102.2000
4,0.0896,0.1101,0.8973,0.8455,0.8714,111.2000
5,0.0305,0.0634,0.9168,0.8723,0.8946,114.0000
6,0.0133,0.0426,0.9146,0.8383,0.8764,131.6000
7,0.0109,0.0266,0.9092,0.8565,0.8829,113.6000
8,0.0080,0.0118,0.9100,0.8646,0.8873,114.2000



Learning rate 3e-05


epoch,loss_intent,loss_relasi,val_f1_intent,val_f1_relasi,val_skor,detik
1,0.5869,0.5265,0.8376,0.3980,0.6178,119.3000
2,0.3051,0.3085,0.8651,0.5621,0.7136,109.7000
3,0.1685,0.2360,0.8865,0.7198,0.8031,112.1000
4,0.0936,0.1461,0.8912,0.7843,0.8378,113.7000
5,0.0462,0.0898,0.9080,0.8982,0.9031,109.1000
6,0.0115,0.0536,0.9148,0.8720,0.8934,110.0000
7,0.0101,0.0351,0.9150,0.8620,0.8885,109.8000
8,0.0021,0.0266,0.9094,0.8936,0.9015,105.2000


## Evaluasi Ujung ke Ujung

Kedua sistem dinilai dengan cara yang sama: intent pesan ditebak model, lalu target disusun dari relasi kandidat.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `susun_target` | Menggabungkan relasi kandidat menjadi target satu pesan. | Intent neutral: hanya relasi neutral (boleh kosong). Offend/defend: hanya relasi offend/defend; bila kosong, `tidak_diketahui` (sama dengan bot di game). |
| `terapkan_ambang` | Aturan minimal satu target. | Pesan offend/defend tanpa target memilih kandidat dengan peluang relasi sesuai intent tertinggi bila ≥ ambang. |
| `AMBANG_KANDIDAT` | Pilihan ambang. | Dipilih dari **validation** dengan intent hasil tebakan, terpisah untuk tiap sistem (sama dengan `nlu_target`). |
| `exact_match` | Semua target satu pesan benar persis. | Metrik utama per pesan; juga dirinci per intent asli dan per kasus. |

In [11]:
# Pilihan ambang aturan minimal satu target (sama dengan nlu_target).
AMBANG_KANDIDAT = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]


# Target satu pesan dari relasi kandidat, disaring sesuai intent tebakan.
def susun_target(kandidat_relasi, intent):
    boleh = {"neutral"} if intent == "neutral" else {"offend", "defend"}
    targets = [{"pemain": k, "relasi": r} for k, r in kandidat_relasi if r in boleh]
    if targets or intent == "neutral":
        return targets
    return [{"pemain": "tidak_diketahui", "relasi": intent}]


# Pesan offend/defend tanpa target: kandidat dengan peluang relasi sesuai intent tertinggi dipilih bila >= ambang.
def terapkan_ambang(pairs, prediksi, proba, intent_pesan, ambang):
    hasil = pd.Series(list(prediksi), index=pairs.index)
    if ambang <= 0:
        return hasil
    for row_id, grup in pairs.assign(prediksi=hasil.values).groupby("row_id"):
        intent = intent_pesan[row_id]
        if intent == "neutral" or grup["prediksi"].isin(["offend", "defend"]).any():
            continue
        peluang_intent = proba.loc[grup.index, intent]
        if peluang_intent.max() >= ambang:
            hasil[peluang_intent.idxmax()] = intent
    return hasil


# Membandingkan target tebakan dengan kunci jawaban per pesan.
def nilai_pesan(rows, pairs, prediksi, intent_pesan):
    pairs = pairs.assign(prediksi=list(prediksi))
    records = []
    for row_id, row in rows.iterrows():
        grup = pairs[pairs["row_id"] == row_id]
        got = {(t["pemain"], t["relasi"]) for t in susun_target(zip(grup["kandidat"], grup["prediksi"]), intent_pesan[row_id])}
        gold = {(t["pemain"], t["relasi"]) for t in row["target"]}
        gk, ok = {x for x in gold if x[0] != "tidak_diketahui"}, {x for x in got if x[0] != "tidak_diketahui"}
        records.append({"row_id": row_id, "case_name": row["case_name"], "teks_chat": row["teks_chat"],
                        "intent_asli": row["label_intent"], "intent_prediksi": intent_pesan[row_id],
                        "target_asli": sorted(gold), "target_prediksi": sorted(got), "benar_persis": gold == got,
                        "tp": len(gk & ok), "fp": len(ok - gk), "fn": len(gk - ok)})
    detail = pd.DataFrame(records)
    tp, fp, fn = detail["tp"].sum(), detail["fp"].sum(), detail["fn"].sum()
    p, r = (tp / (tp + fp) if tp + fp else 0.0), (tp / (tp + fn) if tp + fn else 0.0)
    return {"exact_match": float(detail["benar_persis"].mean()), "target_f1": float(2 * p * r / (p + r) if p + r else 0.0),
            "intent_benar": float((detail["intent_asli"] == detail["intent_prediksi"]).mean()), "detail": detail}


# Ambang dipilih dari validation (seri: ambang terbesar), lalu dipakai di test.
def pilih_ambang(rows, pairs, proba, intent_pesan):
    mentah = proba.idxmax(axis=1)
    skor = [(a, nilai_pesan(rows, pairs, terapkan_ambang(pairs, mentah, proba, intent_pesan, a), intent_pesan)["exact_match"])
            for a in AMBANG_KANDIDAT]
    terbaik = max(e for _, e in skor)
    return max(a for a, e in skor if e == terbaik)


# Ringkasan metrik intent pada sekumpulan chat.
def metrik_intent(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "f1_macro": float(f1_score(y_true, y_pred, labels=INTENT_LABELS, average="macro", zero_division=0)),
            "report": classification_report(y_true, y_pred, labels=INTENT_LABELS, zero_division=0),
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=INTENT_LABELS).tolist()}

### Model Gabungan pada Test

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `intent test` | Test set Model 1 (chat tanpa konteks). | Dibandingkan langsung dengan Model 1. |
| `intent pesan target` | Intent pesan utama di test target. | Dipakai menyusun target ujung ke ujung. |
| `relasi pasangan` | Macro-F1 relasi per pasangan. | Sebelum aturan intent dan ambang. |

In [12]:
# Intent: test Model 1 dan pesan utama dataset target.
def intent_gabungan(teks):
    return [INTENT_LABELS[i] for i in peluang(model_gabungan, list(teks), "intent").argmax(axis=1)]


# Relasi: probabilitas per pasangan tanpa awalan intent.
def proba_relasi_gabungan(pairs):
    return pd.DataFrame(peluang(model_gabungan, pairs["teks"].tolist(), "relasi"), columns=RELASI_LABELS, index=pairs.index)


# Langsung evaluasi model gabungan.
x_test_intent, y_test_intent = split_intent["test"]
hasil_gabungan = {"intent_test": metrik_intent(list(y_test_intent), intent_gabungan(x_test_intent))}
intent_target = {nama: dict(zip(rows.index, intent_gabungan(rows["teks_chat"]))) for nama, rows in split_target.items()
                 if nama != "train"}
proba_val, proba_test = proba_relasi_gabungan(pasangan["validation"]), proba_relasi_gabungan(pasangan["test"])
ambang_gabungan = pilih_ambang(split_target["validation"], pasangan["validation"], proba_val, intent_target["validation"])
prediksi_test = terapkan_ambang(pasangan["test"], proba_test.idxmax(axis=1), proba_test, intent_target["test"], ambang_gabungan)
hasil_gabungan["pesan"] = nilai_pesan(split_target["test"], pasangan["test"], prediksi_test, intent_target["test"])
hasil_gabungan["relasi_f1_macro"] = float(f1_score(pasangan["test"]["relasi"], proba_test.idxmax(axis=1),
                                                   labels=RELASI_LABELS, average="macro", zero_division=0))
hasil_gabungan["ambang"] = ambang_gabungan

print("INTENT (test Model 1)")
print(hasil_gabungan["intent_test"]["report"])
print(f"Relasi macro-F1 pasangan: {hasil_gabungan['relasi_f1_macro']:.4f} | ambang dari validation: {ambang_gabungan:g}")
print(f"Exact match target pesan: {hasil_gabungan['pesan']['exact_match']:.4f} | "
      f"intent pesan target benar: {hasil_gabungan['pesan']['intent_benar']:.4f}")

INTENT (test Model 1)
              precision    recall  f1-score   support

      defend       0.81      0.85      0.83       210
     neutral       0.92      0.90      0.91       387
      offend       0.94      0.94      0.94       526

    accuracy                           0.91      1123
   macro avg       0.89      0.89      0.89      1123
weighted avg       0.91      0.91      0.91      1123

Relasi macro-F1 pasangan: 0.8441 | ambang dari validation: 0.5
Exact match target pesan: 0.7548 | intent pesan target benar: 0.7097


### Menyimpan Model Gabungan

| Berkas | Isi |
| --- | --- |
| `model.safetensors` | Bobot encoder dan kedua kepala. |
| `config.json`, `tokenizer.json`, `vocab.txt` | Konfigurasi encoder dan tokenizer IndoBERT. |
| `gabungan_config.json` | Label, panjang token, ambang, pengaturan training, dan pemenang learning rate. |
| `metrics.json`, `test_predictions.csv` | Metrik test dan prediksi per pesan untuk analisis kesalahan. |

In [13]:
# Menyimpan model, tokenizer, info label, metrik, dan prediksi test.
if LATIH_GABUNGAN:
    info = {"intent_labels": INTENT_LABELS, "relasi_labels": RELASI_LABELS, "max_length": PENGATURAN["max_length"],
            "ambang": ambang_gabungan, "pengaturan": {k: v for k, v in PENGATURAN.items() if k != "device"},
            "learning_rate": pemenang["learning_rate"], "epoch_terbaik": pemenang["terbaik"]["epoch"],
            "durasi_training_detik": round(durasi_training, 1), "transformers_version": transformers.__version__,
            "torch_version": torch.__version__, "pair_builder": {"common_words": sorted(COMMON_WORDS),
            "jumlah_pemain_minimum": JUMLAH_PEMAIN_MINIMUM}}
    model_gabungan.simpan(MODEL_GABUNGAN_DIR, info)
    tokenizer.save_pretrained(MODEL_GABUNGAN_DIR)
    metrik_simpan = {"intent_test": {k: v for k, v in hasil_gabungan["intent_test"].items()},
                     "relasi_f1_macro": hasil_gabungan["relasi_f1_macro"], "ambang": ambang_gabungan,
                     "pesan": {k: v for k, v in hasil_gabungan["pesan"].items() if k != "detail"},
                     "riwayat": [{"learning_rate": t["learning_rate"], "riwayat": t["riwayat"]} for t in hasil_trial]}
    (MODEL_GABUNGAN_DIR / "metrics.json").write_text(json.dumps(metrik_simpan, indent=2), encoding="utf-8")
    hasil_gabungan["pesan"]["detail"].to_csv(MODEL_GABUNGAN_DIR / "test_predictions.csv", index=False)
    print("Model gabungan tersimpan:", MODEL_GABUNGAN_DIR)
else:
    print("Penyimpanan dilewati: LATIH_GABUNGAN = False.")

Model gabungan tersimpan: models\gabungan\nlu_gabungan_transformer


## Perbandingan dengan Pipeline Dua Model

Model 1 dan Model 2 dimuat dari folder hasil `nlu_baru` dan `nlu_target`, lalu dinilai dengan fungsi dan data yang sama.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `pipeline (intent tebakan)` | Kondisi nyata di game. | Model 1 menebak intent, Model 2 menerima intent itu di awal teks pasangan. |
| `pipeline (intent kunci)` | Batas atas pipeline. | Model 2 menerima intent asli, seperti laporan di `nlu_target`; tidak mungkin terjadi di game. |
| `ukuran` | Jumlah dan besar berkas bobot. | Pipeline memuat dua IndoBERT; model gabungan satu. |

In [14]:
# Memuat model klasifikasi tersimpan (Model 1 atau Model 2).
def muat_klasifikasi(folder):
    tok = AutoTokenizer.from_pretrained(folder, local_files_only=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(folder, local_files_only=True).to(DEVICE).eval()
    return tok, mdl


# Probabilitas kelas dari model klasifikasi tersimpan, dengan urutan label model itu.
@torch.inference_mode()
def peluang_klasifikasi(tok, mdl, teks, max_length):
    hasil = []
    for k in range(0, len(teks), PENGATURAN["eval_batch_size"]):
        enc = tok(list(teks[k:k + PENGATURAN["eval_batch_size"]]), truncation=True, max_length=max_length,
                  padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=PRESISI, enabled=PRESISI is not None):
            hasil.append(torch.softmax(mdl(**enc).logits.float(), dim=-1).cpu().numpy())
    labels = [mdl.config.id2label[i] for i in range(mdl.config.num_labels)]
    return pd.DataFrame(np.concatenate(hasil), columns=labels)


# Langsung nilai pipeline dengan data dan fungsi yang sama.
tok1, model_1 = muat_klasifikasi(MODEL_1_DIR)
tok2, model_2 = muat_klasifikasi(MODEL_2_DIR)
intent_1 = lambda teks: peluang_klasifikasi(tok1, model_1, list(teks), 128).idxmax(axis=1).tolist()
hasil_pipeline = {"intent_test": metrik_intent(list(y_test_intent), intent_1(x_test_intent))}


# Pipeline: pasangan dengan awalan intent tertentu → relasi Model 2 → ambang dari validation → target.
def nilai_pipeline(intent_pesan_per_bagian):
    pairs = {nama: bentuk_pasangan(split_target[nama], COMMON_WORDS, JUMLAH_PEMAIN_MINIMUM, intent_pesan_per_bagian[nama])
             for nama in ("validation", "test")}
    proba = {nama: peluang_klasifikasi(tok2, model_2, frame["teks"].tolist(), 512).set_index(frame.index)
             for nama, frame in pairs.items()}
    ambang = pilih_ambang(split_target["validation"], pairs["validation"], proba["validation"],
                          intent_pesan_per_bagian["validation"])
    prediksi = terapkan_ambang(pairs["test"], proba["test"].idxmax(axis=1), proba["test"],
                               intent_pesan_per_bagian["test"], ambang)
    return nilai_pesan(split_target["test"], pairs["test"], prediksi, intent_pesan_per_bagian["test"]), ambang


intent_pipeline = {nama: dict(zip(split_target[nama].index, intent_1(split_target[nama]["teks_chat"])))
                   for nama in ("validation", "test")}
intent_kunci = {nama: dict(zip(split_target[nama].index, split_target[nama]["label_intent"]))
                for nama in ("validation", "test")}
hasil_pipeline["pesan"], hasil_pipeline["ambang"] = nilai_pipeline(intent_pipeline)
pesan_kunci, ambang_kunci = nilai_pipeline(intent_kunci)
print("Pipeline siap dinilai. Ambang pipeline (intent tebakan / intent kunci):", hasil_pipeline["ambang"], ambang_kunci)

Pipeline siap dinilai. Ambang pipeline (intent tebakan / intent kunci): 0.5 0.5


### Tabel Perbandingan

| Kolom | Arti |
| --- | --- |
| `Intent acc / macro-F1` | Pada test Model 1 (chat tanpa konteks). |
| `Exact match target` | Semua target satu pesan benar persis, ujung ke ujung. |
| `EM offend / defend / neutral` | Exact match per intent asli pesan. |
| `Model / ukuran` | Jumlah IndoBERT yang dimuat dan besar bobotnya. |

In [15]:
# Ukuran bobot tersimpan dalam MB.
ukuran = lambda *folders: round(sum((Path(f) / "model.safetensors").stat().st_size for f in folders) / 2**20)


# Satu baris tabel perbandingan.
def baris(nama, intent, pesan, n_model, mb):
    detail = pesan["detail"]
    per_intent = detail.groupby("intent_asli")["benar_persis"].mean()
    return {"Sistem": nama, "Intent acc": intent["accuracy"] if intent else None,
            "Intent macro-F1": intent["f1_macro"] if intent else None,
            "Intent pesan target": pesan["intent_benar"], "Exact match target": pesan["exact_match"],
            "Target F1": pesan["target_f1"], **{f"EM {k}": per_intent.get(k) for k in ("offend", "defend", "neutral")},
            "Model": n_model, "Ukuran (MB)": mb}


perbandingan = pd.DataFrame([
    baris("Pipeline (intent tebakan)", hasil_pipeline["intent_test"], hasil_pipeline["pesan"], 2, ukuran(MODEL_1_DIR, MODEL_2_DIR)),
    baris("Model gabungan", hasil_gabungan["intent_test"], hasil_gabungan["pesan"], 1, ukuran(MODEL_GABUNGAN_DIR)),
    baris("Pipeline (intent kunci, batas atas)", None, pesan_kunci, 2, ukuran(MODEL_1_DIR, MODEL_2_DIR)),
])
display(perbandingan.style.format(precision=4, na_rep="-").hide(axis="index"))
perbandingan.to_csv(MODEL_GABUNGAN_DIR / "perbandingan_pipeline.csv", index=False)

# Exact match per kasus untuk kedua sistem.
display(pd.DataFrame({
    "Pipeline": hasil_pipeline["pesan"]["detail"].groupby("case_name")["benar_persis"].mean(),
    "Gabungan": hasil_gabungan["pesan"]["detail"].groupby("case_name")["benar_persis"].mean(),
}).style.format("{:.2f}").background_gradient(axis=None, cmap="RdYlGn", vmin=0, vmax=1))

Sistem,Intent acc,Intent macro-F1,Intent pesan target,Exact match target,Target F1,EM offend,EM defend,EM neutral,Model,Ukuran (MB)
Pipeline (intent tebakan),0.9012,0.8837,0.5742,0.4968,0.4899,0.7419,0.2059,0.6800,2,949
Model gabungan,0.9074,0.8923,0.7097,0.7548,0.8054,0.7742,0.8235,0.5200,1,475
"Pipeline (intent kunci, batas atas)",-,-,1.0000,0.8581,0.8986,0.8387,0.8676,0.8800,2,949


,Pipeline,Gabungan
case_name,,
agreement,0.60,0.80
ambiguous,0.40,0.60
explicit,0.40,0.80
first_message,0.80,1.00
first_message_self,0.60,0.80
first_message_unknown,0.80,0.80
interruption,0.40,0.60
memory_outside_window,0.40,1.00
mixed_relations,0.60,0.60


### Pesan yang Hasilnya Berbeda

Daftar pesan test yang benar di satu sistem tetapi salah di sistem lain, untuk analisis kesalahan.

In [16]:
# Pesan yang benar di satu sistem dan salah di sistem lain.
beda = hasil_pipeline["pesan"]["detail"][["row_id", "case_name", "teks_chat", "target_asli", "intent_prediksi",
                                          "target_prediksi", "benar_persis"]].merge(
    hasil_gabungan["pesan"]["detail"][["row_id", "intent_prediksi", "target_prediksi", "benar_persis"]],
    on="row_id", suffixes=("_pipeline", "_gabungan"))
beda = beda[beda["benar_persis_pipeline"] != beda["benar_persis_gabungan"]]
print(f"{len(beda)} pesan berbeda | hanya pipeline benar: {int(beda['benar_persis_pipeline'].sum())} | "
      f"hanya gabungan benar: {int(beda['benar_persis_gabungan'].sum())}")
display(beda.drop(columns="row_id").style.hide(axis="index").set_properties(
    subset=["teks_chat"], **{"white-space": "normal", "max-width": "380px"}))

60 pesan berbeda | hanya pipeline benar: 10 | hanya gabungan benar: 50


case_name,teks_chat,target_asli,intent_prediksi_pipeline,target_prediksi_pipeline,benar_persis_pipeline,intent_prediksi_gabungan,target_prediksi_gabungan,benar_persis_gabungan
ambiguous,Aku curiga dia Hitman yang sengaja mengubah arah tuduhan.,"[('tidak_diketahui', 'offend')]",offend,"[('tidak_diketahui', 'offend')]",True,offend,"[('Jojo', 'offend')]",False
outside_window_unknown,Menurutku dia belum pantas dituduh hanya karena cepat membela.,"[('tidak_diketahui', 'defend')]",offend,"[('tidak_diketahui', 'offend')]",False,defend,"[('tidak_diketahui', 'defend')]",True
stance_change,"Oh kalau kenyataannya begitu aku tarik tuduhanku tadi, dia sangat wajar telat membalas karena masalah teknis jaringannya.","[('Zaki', 'defend')]",offend,"[('Zaki', 'offend')]",False,defend,"[('Zaki', 'defend')]",True
reference_end,"Jangan langsung menuduh dia. Bisa saja pembelaannya masuk akal, mari baca lagi alasannya.","[('Rudi', 'defend')]",offend,"[('Rudi', 'offend')]",False,defend,"[('Rudi', 'defend')]",True
reference_middle,Benar kata Sinta. Dia belum pantas dituduh sebagai Hitman hanya karena mengubah dugaan.,"[('Nanda', 'defend')]",offend,"[('Nanda', 'offend')]",False,offend,"[('Nanda', 'defend')]",True
short_history_1,"Kamu tidak bisa mengajak kami vote Fira hanya bermodalkan firasat saja, dia memiliki hak untuk menjelaskan posisinya.","[('Fira', 'defend')]",offend,"[('Fira', 'offend')]",False,offend,"[('Fira', 'defend')]",True
short_history_3,Citra belum pantas dituduh hanya dari jeda chat. Penyebabnya bisa masalah koneksi.,"[('Citra', 'defend')]",offend,"[('Citra', 'offend')]",False,offend,"[('Citra', 'defend')]",True
quotation,"Maman bilang 'Dia pasti Hitman', tetapi aku tidak setuju. Lala belum pantas dituduh dari perubahan dugaan itu.","[('Lala', 'defend')]",offend,"[('Lala', 'offend')]",False,offend,"[('Lala', 'defend')]",True
short_history_3,Sandi belum pantas dituduh sebagai Hitman hanya karena menarik ajakan vote yang belum berdasar.,"[('Sandi', 'defend')]",offend,"[('Sandi', 'offend')]",False,offend,"[('Sandi', 'defend')]",True
first_message,"Aku belum punya alasan mencurigai Lila, jadi jangan langsung ajak vote Lila.","[('Lila', 'defend')]",offend,"[('Lila', 'offend')]",False,offend,"[('Lila', 'defend')]",True


## Catatan Evaluasi

- **Adil:** kedua sistem memakai data, split, teks pasangan, ruang learning rate, dan aturan seleksi yang sama; ambang dipilih dari validation untuk masing-masing.
- **Intent kunci** hanya batas atas pipeline. Di game intent selalu hasil tebakan.
- **Konteks:** anotasi pesan konteks memakai label dataset untuk kedua sistem, sama dengan evaluasi `nlu_target`. Di game, anotasi ini berasal dari tebakan sebelumnya.
- **Varian nama:** game merata-ratakan intent dari tiga varian nama (`varian_nama_intent`). Evaluasi ini memakai teks asli untuk kedua sistem.
- **Ukuran test target kecil** (±155 pesan): selisih 1 pesan ≈ 0,6 poin exact match. Selisih kecil belum tentu nyata.